# Pipeline Conversor de Meritocracia — V3

Melhorias:
- Log com schema fixo.
- Histórico dos resultados em tabela Delta.
- Validações mais fortes.
- Saída e erros separados em diretórios do Volume.
- `run_id` para rastrear cada execução.
- `try/except` com log de erro.


In [0]:
%pip install openpyxl


In [0]:
dbutils.library.restartPython()


In [0]:
import os
import json
import uuid
import traceback
import pandas as pd

from datetime import datetime
from collections import OrderedDict

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType,
    TimestampType
)


In [0]:
def transformar_para_hierarquia(df):
    df = df.copy()

    df["cGrpCobrMotorDecis"] = df["cGrpCobrMotorDecis"].ffill()
    df["cCanalCobrMotorDecis"] = df["cCanalCobrMotorDecis"].ffill()
    df["DISTRIBUIÇÃO cCanalCobrMotorDecis"] = df["DISTRIBUIÇÃO cCanalCobrMotorDecis"].ffill()

    resultado = {"perfis": {}}
    avisos_soma = []

    for perfil, grupo_perfil in df.groupby("cGrpCobrMotorDecis", sort=False):
        id_portfolio = "BCO"

        dados_portfolio = OrderedDict()
        dados_portfolio["nome"] = id_portfolio
        dados_portfolio["canais"] = OrderedDict()

        resultado["perfis"][perfil] = {
            "portfolios": {
                id_portfolio: dados_portfolio
            }
        }

        df_canais_unicos = grupo_perfil.drop_duplicates("cCanalCobrMotorDecis")
        soma_canais = df_canais_unicos["DISTRIBUIÇÃO cCanalCobrMotorDecis"].sum()

        if soma_canais != 100:
            avisos_soma.append(f"PERFIL: {perfil} | SOMA CANAIS: {soma_canais}%")

        for canal, grupo_canal in grupo_perfil.groupby("cCanalCobrMotorDecis", sort=False):
            percentual_canal = int(grupo_canal["DISTRIBUIÇÃO cCanalCobrMotorDecis"].iloc[0])
            soma_assessorias = grupo_canal["DISTRIBUIÇÃO"].sum()

            if soma_assessorias != 100:
                avisos_soma.append(
                    f"PERFIL: {perfil} | CANAL: {canal} | SOMA ASSESSORIAS: {soma_assessorias}%"
                )

            dados_canal = OrderedDict()
            dados_canal["percentual"] = percentual_canal
            dados_canal["assessorias"] = OrderedDict()

            dados_portfolio["canais"][canal] = dados_canal

            for _, linha in grupo_canal.iterrows():
                assessoria = str(linha["cDecisAssesMotorDecis"])
                percentual_assessoria = int(linha["DISTRIBUIÇÃO"])

                dados_canal["assessorias"][assessoria] = {
                    "percentual": percentual_assessoria
                }

    return resultado, avisos_soma


In [0]:
dbutils.widgets.text(
    "arquivo_entrada",
    "/Volumes/workspace/default/planilha-exemplo/PLANILHA EXEMPLO.xlsx",
    "Caminho da planilha de entrada"
)

dbutils.widgets.text(
    "diretorio_saida",
    "/Volumes/workspace/default/planilha-exemplo/saida",
    "Diretório onde o JSON será salvo"
)

dbutils.widgets.text(
    "diretorio_erro",
    "/Volumes/workspace/default/planilha-exemplo/erros",
    "Diretório onde os arquivos de erro serão salvos"
)

dbutils.widgets.text(
    "nome_saida",
    "saida_meritocracia.json",
    "Nome base do arquivo JSON de saída"
)

dbutils.widgets.dropdown(
    "falhar_com_avisos",
    "false",
    ["false", "true"],
    "Falhar o pipeline se houver avisos?"
)

dbutils.widgets.text(
    "tabela_log",
    "log_conversor_meritocracia",
    "Tabela Delta de log do pipeline"
)

dbutils.widgets.text(
    "tabela_resultados",
    "historico_conversor_meritocracia",
    "Tabela Delta com histórico dos JSONs gerados"
)

arquivo_entrada = dbutils.widgets.get("arquivo_entrada").strip()
diretorio_saida = dbutils.widgets.get("diretorio_saida").strip()
diretorio_erro = dbutils.widgets.get("diretorio_erro").strip()
nome_saida_base = dbutils.widgets.get("nome_saida").strip()
falhar_com_avisos = dbutils.widgets.get("falhar_com_avisos").strip().lower() == "true"
tabela_log = dbutils.widgets.get("tabela_log").strip()
tabela_resultados = dbutils.widgets.get("tabela_resultados").strip()

if not nome_saida_base:
    nome_saida_base = "saida_meritocracia.json"

run_id = str(uuid.uuid4())
timestamp_execucao = datetime.now()
timestamp_arquivo = timestamp_execucao.strftime("%Y%m%d_%H%M%S")

nome_sem_extensao = os.path.splitext(nome_saida_base)[0]
nome_saida = f"{nome_sem_extensao}_{timestamp_arquivo}.json"

print("Run ID:", run_id)
print("Arquivo de entrada:", arquivo_entrada)
print("Diretório de saída:", diretorio_saida)
print("Diretório de erro:", diretorio_erro)
print("Nome final do JSON:", nome_saida)
print("Falhar com avisos:", falhar_com_avisos)
print("Tabela de log:", tabela_log)
print("Tabela de resultados:", tabela_resultados)


In [0]:
COLUNAS_OBRIGATORIAS = [
    "cGrpCobrMotorDecis",
    "cCanalCobrMotorDecis",
    "DISTRIBUIÇÃO cCanalCobrMotorDecis",
    "cDecisAssesMotorDecis",
    "DISTRIBUIÇÃO"
]

COLUNAS_PERCENTUAIS = [
    "DISTRIBUIÇÃO cCanalCobrMotorDecis",
    "DISTRIBUIÇÃO"
]

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("arquivo_entrada", StringType(), True),
    StructField("arquivo_saida", StringType(), True),
    StructField("arquivo_erro", StringType(), True),
    StructField("linhas_processadas", LongType(), True),
    StructField("quantidade_avisos", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("mensagem_erro", StringType(), True),
    StructField("data_execucao", TimestampType(), True)
])

RESULTADO_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("arquivo_entrada", StringType(), True),
    StructField("arquivo_saida", StringType(), True),
    StructField("nome_arquivo_saida", StringType(), True),
    StructField("linhas_processadas", LongType(), True),
    StructField("perfis_processados", IntegerType(), True),
    StructField("quantidade_avisos", IntegerType(), True),
    StructField("json_conteudo", StringType(), True),
    StructField("data_execucao", TimestampType(), True)
])


def criar_diretorios():
    if not diretorio_saida:
        raise ValueError("O parâmetro 'diretorio_saida' não foi preenchido.")

    if not diretorio_erro:
        raise ValueError("O parâmetro 'diretorio_erro' não foi preenchido.")

    os.makedirs(diretorio_saida, exist_ok=True)
    os.makedirs(diretorio_erro, exist_ok=True)


def carregar_arquivo(caminho):
    if not caminho:
        raise ValueError("O parâmetro 'arquivo_entrada' não foi preenchido.")

    if not os.path.exists(caminho):
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    if caminho.lower().endswith(".csv"):
        return pd.read_csv(caminho, encoding="latin1")

    if caminho.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(caminho)

    raise ValueError("Formato não suportado. Use .csv, .xlsx ou .xls")


def preparar_e_validar_dataframe(df):
    df = df.copy()

    colunas_faltando = [
        coluna for coluna in COLUNAS_OBRIGATORIAS
        if coluna not in df.columns
    ]

    if colunas_faltando:
        raise ValueError(f"Colunas obrigatórias ausentes: {colunas_faltando}")

    df["cGrpCobrMotorDecis"] = df["cGrpCobrMotorDecis"].ffill()
    df["cCanalCobrMotorDecis"] = df["cCanalCobrMotorDecis"].ffill()
    df["DISTRIBUIÇÃO cCanalCobrMotorDecis"] = df["DISTRIBUIÇÃO cCanalCobrMotorDecis"].ffill()

    df = df.dropna(how="all")

    problemas_nulos = {}

    for coluna in COLUNAS_OBRIGATORIAS:
        qtd_nulos = int(df[coluna].isna().sum())
        if qtd_nulos > 0:
            problemas_nulos[coluna] = qtd_nulos

    if problemas_nulos:
        raise ValueError(f"Existem valores nulos em colunas obrigatórias: {problemas_nulos}")

    for coluna in COLUNAS_PERCENTUAIS:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

        if df[coluna].isna().any():
            linhas_invalidas = df[df[coluna].isna()].index.tolist()
            raise ValueError(
                f"A coluna '{coluna}' possui valores não numéricos nas linhas: {linhas_invalidas}"
            )

        valores_invalidos = df[(df[coluna] < 0) | (df[coluna] > 100)]

        if not valores_invalidos.empty:
            linhas_invalidas = valores_invalidos.index.tolist()
            raise ValueError(
                f"A coluna '{coluna}' possui percentuais fora do intervalo 0 a 100 nas linhas: {linhas_invalidas}"
            )

    return df


def salvar_json_em_volume(json_str):
    arquivo_saida = f"{diretorio_saida.rstrip('/')}/{nome_saida}"

    with open(arquivo_saida, "w", encoding="utf-8") as f:
        f.write(json_str)

    return arquivo_saida


def salvar_arquivo_erro(mensagem_erro, stacktrace):
    nome_arquivo_erro = f"erro_conversor_{timestamp_arquivo}.txt"
    arquivo_erro = f"{diretorio_erro.rstrip('/')}/{nome_arquivo_erro}"

    conteudo_erro = (
        f"run_id: {run_id}\n"
        f"data_execucao: {datetime.now().isoformat()}\n"
        f"arquivo_entrada: {arquivo_entrada}\n"
        f"mensagem_erro: {mensagem_erro}\n\n"
        f"stacktrace:\n{stacktrace}"
    )

    with open(arquivo_erro, "w", encoding="utf-8") as f:
        f.write(conteudo_erro)

    return arquivo_erro


def registrar_log(
    arquivo_saida,
    arquivo_erro,
    linhas_processadas,
    quantidade_avisos,
    status,
    mensagem_erro=""
):
    dados_log = [(
        run_id,
        arquivo_entrada,
        arquivo_saida,
        arquivo_erro,
        int(linhas_processadas),
        int(quantidade_avisos),
        status,
        mensagem_erro,
        datetime.now()
    )]

    df_log = spark.createDataFrame(dados_log, schema=LOG_SCHEMA)

    (
        df_log
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela_log)
    )

    return df_log


def registrar_resultado(
    arquivo_saida,
    json_str,
    linhas_processadas,
    perfis_processados,
    quantidade_avisos
):
    dados_resultado = [(
        run_id,
        arquivo_entrada,
        arquivo_saida,
        nome_saida,
        int(linhas_processadas),
        int(perfis_processados),
        int(quantidade_avisos),
        json_str,
        datetime.now()
    )]

    df_resultado = spark.createDataFrame(dados_resultado, schema=RESULTADO_SCHEMA)

    (
        df_resultado
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela_resultados)
    )

    return df_resultado


In [0]:
df = None
avisos = []
arquivo_saida = ""
arquivo_erro = ""

try:
    print("Iniciando pipeline_conversor_meritocracia V3...")

    print("\n1. Criando/validando diretórios...")
    criar_diretorios()
    print("Diretórios OK.")

    print("\n2. Carregando planilha...")
    df_original = carregar_arquivo(arquivo_entrada)
    print("Arquivo carregado com sucesso!")
    print(f"Quantidade de linhas originais: {len(df_original)}")
    print(f"Quantidade de colunas: {len(df_original.columns)}")

    print("\n3. Preparando e validando dados...")
    df = preparar_e_validar_dataframe(df_original)
    print("Validação concluída com sucesso!")
    print(f"Quantidade de linhas processáveis: {len(df)}")

    print("\n4. Convertendo para JSON...")
    json_data, avisos = transformar_para_hierarquia(df)

    json_str = json.dumps(
        json_data,
        indent=2,
        ensure_ascii=False,
        sort_keys=False
    )

    perfis_processados = len(json_data.get("perfis", {}))

    print("Conversão concluída!")
    print(f"Perfis processados: {perfis_processados}")

    if avisos:
        print("\nAvisos encontrados:")

        for aviso in avisos:
            print(f"- {aviso}")

        if falhar_com_avisos:
            raise ValueError(
                "O pipeline foi configurado para falhar quando houver avisos. "
                f"Avisos encontrados: {avisos}"
            )
    else:
        print("Nenhum aviso encontrado.")

    print("\n5. Salvando JSON no Volume...")
    arquivo_saida = salvar_json_em_volume(json_str)
    print("JSON salvo com sucesso!")
    print(f"Caminho do arquivo salvo: {arquivo_saida}")

    print("\n6. Lendo conteúdo salvo para conferência...")

    with open(arquivo_saida, "r", encoding="utf-8") as f:
        conteudo_salvo = f.read()

    print("\nConteúdo do arquivo salvo:")
    print(conteudo_salvo)

    status_execucao = "SUCESSO_COM_AVISO" if avisos else "SUCESSO"

    print("\n7. Registrando log...")
    df_log = registrar_log(
        arquivo_saida=arquivo_saida,
        arquivo_erro="",
        linhas_processadas=len(df),
        quantidade_avisos=len(avisos),
        status=status_execucao,
        mensagem_erro=""
    )

    print("Log salvo com sucesso.")
    display(df_log)

    print("\n8. Registrando histórico do resultado...")
    df_resultado = registrar_resultado(
        arquivo_saida=arquivo_saida,
        json_str=json_str,
        linhas_processadas=len(df),
        perfis_processados=perfis_processados,
        quantidade_avisos=len(avisos)
    )

    print("Histórico do resultado salvo com sucesso.")
    display(df_resultado)

    print("\nPipeline finalizado com sucesso!")
    print("Status:", status_execucao)
    print("Run ID:", run_id)

except Exception as erro:
    mensagem_erro = str(erro)
    stacktrace = traceback.format_exc()
    linhas_processadas = len(df) if df is not None else 0

    print("Pipeline finalizado com erro.")
    print("Erro:", mensagem_erro)

    try:
        criar_diretorios()
        arquivo_erro = salvar_arquivo_erro(mensagem_erro, stacktrace)
        print(f"Arquivo de erro salvo em: {arquivo_erro}")

    except Exception as erro_arquivo:
        print("Não foi possível salvar o arquivo de erro.")
        print("Erro ao salvar arquivo de erro:", str(erro_arquivo))

    try:
        df_log = registrar_log(
            arquivo_saida=arquivo_saida,
            arquivo_erro=arquivo_erro,
            linhas_processadas=linhas_processadas,
            quantidade_avisos=len(avisos),
            status="ERRO",
            mensagem_erro=mensagem_erro
        )

        print("Log de erro salvo com sucesso.")
        display(df_log)

    except Exception as erro_log:
        print("Não foi possível salvar o log de erro.")
        print("Erro ao registrar log:", str(erro_log))

    raise


In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {tabela_log}
    ORDER BY data_execucao DESC
    LIMIT 20
"""))


In [0]:
display(spark.sql(f"""
    SELECT
        run_id,
        arquivo_entrada,
        arquivo_saida,
        nome_arquivo_saida,
        linhas_processadas,
        perfis_processados,
        quantidade_avisos,
        data_execucao
    FROM {tabela_resultados}
    ORDER BY data_execucao DESC
    LIMIT 20
"""))


In [0]:
ultimo_log = spark.sql(f"""
    SELECT *
    FROM {tabela_log}
    ORDER BY data_execucao DESC
    LIMIT 1
""").collect()[0]

print("Último run_id:", ultimo_log["run_id"])
print("Status:", ultimo_log["status"])
print("Arquivo de saída:", ultimo_log["arquivo_saida"])
print("Arquivo existe?", os.path.exists(ultimo_log["arquivo_saida"]) if ultimo_log["arquivo_saida"] else False)

if ultimo_log["arquivo_erro"]:
    print("Arquivo de erro:", ultimo_log["arquivo_erro"])
    print("Arquivo de erro existe?", os.path.exists(ultimo_log["arquivo_erro"]))
